In [ ]:
import os, sys
import pickle

import matplotlib_inline
sys.path.append("../")

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import numpy as np
import pysm3
import pysm3.units as u
import astropy.units as un
import astropy.constants as const
from tqdm import tqdm
import pymaster as nmt

matplotlib_inline.backend_inline.set_matplotlib_formats('retina')

%matplotlib inline
%load_ext autoreload
%autoreload 2
# Load plot settings
import healpy as hp
from plot_params import params
pylab.rcParams.update(params)

cols_default = plt.rcParams['axes.prop_cycle'].by_key()['color']

In [ ]:
binning = nmt.NmtBin.from_nside_linear(2048, 10)
TCMB_eV = 0.000235264 
x = 2 * np.pi * 0.000232349 /TCMB_eV
prefac = TCMB_eV * x**2 / (1- np.exp(-x))
mccarthy_Cls = np.load("/usr3/graduate/ebaker/dark_photon_constraints/ilc/pyilc_files/mccarthy_Cls.npy")
planck_Cls = np.load("/projectnb/darkcosmo/dark_photon_project/21cmfast_cache/pyilc_Planck/clean_auto_Cls_binning10.npy")
mock_planck_Cls = np.load("/projectnb/darkcosmo/dark_photon_project/21cmfast_cache/pyilc_Planck/clean_auto_Cls_binning10.npy")
small_beam = np.load("/projectnb/darkcosmo/dark_photon_project/21cmfast_cache/pyilc_planck_small_beam/clean_auto_Cls_binning10.npy")
simple_Planck_dust = np.load("/projectnb/darkcosmo/dark_photon_project/21cmfast_cache/pyilc_Planck_simple_dust/clean_auto_Cls_binning10.npy")
planck_no_pt_src = np.load("/projectnb/darkcosmo/dark_photon_project/21cmfast_cache/pyilc_Planck_no_pt_srcs/clean_auto_Cls_binning10.npy")
radio = np.load("/projectnb/darkcosmo/dark_photon_project/21cmfast_cache/pyilc_alt_pt_src_100mJy/clean_auto_Cls_binning10.npy")

In [ ]:
ls = np.arange(1,3*nside)
loaded_ls = np.load("../halo_data/correct_Cls/ls.npy")
dp_ps = np.load("/home/bakerem/dark_photon_21cm_constraints/halo_data/correct_Cls/Cl_PP_mA5.623e-13.npy")
interp_power = np.interp(ls, loaded_ls, dp_ps,)
var = np.sum((2*ls + 1) / (4 * np.pi) * interp_power) /((nu_centers * const.h).to(un.eV))**2
rms = (1e-7)**4 * np.sqrt(var)
# x = 2 * np.pi * ((nu_centers * const.h).to(un.eV) / (2.73 * u.K * const.k_B).to(un.eV))
# dp_rms_K_CMB = (2.73 * u.K_CMB) * rms * (1e-7)**2 / (nu_centers * const. h).to(un.eV).value * (1- np.exp(-x))/x
# dp_rms_uK_RJ = u.uK_RJ * np.array([dp_rms_K_CMB[i].to(u.uK_RJ, equivalencies=u.cmb_equivalencies(nu_centers[i])).value for i in range(len(nu_centers))])


In [ ]:
cmb_mono = (2.73 * u.K_CMB).to(u.uK_RJ, equivalencies=u.cmb_equivalencies(nu_centers))

In [ ]:
ls = np.arange(1,3*nside)
loaded_ls = np.load("../halo_data/correct_Cls/ls.npy")
dp_ps = np.load("/home/bakerem/dark_photon_21cm_constraints/halo_data/correct_Cls/Cl_PP_mA5.623e-13.npy")
interp_power = np.interp(ls, loaded_ls, dp_ps,)
var = np.sum((2*ls + 1) / (4 * np.pi) * interp_power) /((nu_centers * const.h).to(un.eV))**2
rms = (1e-7)**2 * np.sqrt(var) * cmb_mono

In [ ]:
# mask = hp.ud_grade(hp.read_map("/usr3/graduate/ebaker/dark_photon_constraints/ilc/pyilc_files/roman_mask_20deg.fits"), 256)
mask = hp.ud_grade(hp.read_map("../halo_data/sed_checks/roman_mask_20deg.fits"), 256)
mccarthy_mask = hp.ud_grade(hp.read_map("../halo_data/sed_checks/mccarthy_mask.fits"), 256)
mask = np.where(mask == 0 ,np.nan, 1)

In [ ]:
pt_srcs = np.load("../halo_data/sed_checks/Tb_nu_map256.npy") * u.K_RJ # Point source maps at each frequency in #K_RJ
pt_srcs = np.swapaxes(pt_srcs, 0, 1)
pt_srcs = pt_srcs.to(u.uK_RJ)

In [ ]:
# first generate maps at the radio frequencies we care about
nside = 256
log2nside = int(np.log2(nside))

# nu_centers = np.geomspace(100, 500e3, 30) # MHz
# generate everything else
full_ls = np.arange(3 * nside)

# all of this is in uK_RJ
# nu_centers = nu_centers * u.MHz
freefreesky = pysm3.Sky(nside=nside, preset_strings=["f1"])
freefree_maps = [
    freefreesky.get_emission(freq)[0,:] for freq in nu_centers
]  # Get free-free maps for each frequency
freefree_maps = np.array( freefree_maps )  * u.uK_RJ
freefree_maps = freefree_maps - np.mean(freefree_maps, axis=1, keepdims=True)

syncsky = pysm3.Sky(nside=nside, preset_strings=["s1"])
sync_maps = [
    syncsky.get_emission(freq)[0,:] for freq in nu_centers
]  # Get free-free maps for each frequency
sync_maps = np.array( sync_maps)  * u.uK_RJ
sync_maps = sync_maps - np.mean(sync_maps, axis=1, keepdims=True)

cmbsky = pysm3.Sky(nside=nside, preset_strings=["c3"])
cmb_maps = [
    cmbsky.get_emission(freq)[0,:] for freq in nu_centers
]
cmb_maps = np.array( cmb_maps ) * u.uK_RJ
cmb_maps = cmb_maps - np.mean(cmb_maps, axis=1, keepdims=True)

  # 2.73 K is the CMB monopole temperature; result is in uK_RJ
Tgamma0 = 2.73  # K



In [ ]:
from tqdm import tqdm
from astropy.config import set_temp_cache, get_cache_dir_path
dustsky = pysm3.Sky(nside=nside, preset_strings=["d1"])
dust_maps = [
    dustsky.get_emission(freq)[0,:] if freq.to(u.GHz) > 1 * u.GHz else np.zeros(hp.nside2npix(nside))*u.uK_RJ for freq in tqdm(nu_centers)
]  # Get free-free maps for each frequency)

dust_maps = np.array(dust_maps) * u.uK_RJ
dust_maps = dust_maps - np.mean(dust_maps, axis=1, keepdims=True)

    
amesky = pysm3.Sky(nside=nside, preset_strings=["a1"])
ame_maps = [
    amesky.get_emission(freq)[0,:] if freq.to(u.GHz) > 1 * u.GHz else np.zeros(hp.nside2npix(nside))*u.K_RJ for freq in tqdm(nu_centers)
]  # Get free-free maps for each frequency)
ame_maps = np.array(ame_maps) * u.uK_RJ
ame_maps = ame_maps - np.mean(ame_maps, axis=1, keepdims=True)


# tszsky = pysm3.Sky(nside=nside, preset_strings=["tsz1"])
# tsz_maps = [
#     tszsky.get_emission(freq)[0,:] if freq.to(u.GHz) > 1 * u.GHz else np.zeros(hp.nside2npix(nside))*u.K_RJ for freq in tqdm(nu_centers)
# ]  # Get free-free maps for each frequency)
# tsz_maps = np.array(tsz_maps) * u.uK_RJ
# tsz_maps = tsz_maps - np.mean(tsz_maps, axis=1, keepdims=True)

cosky = pysm3.Sky(nside=nside, preset_strings=["co1"])
co_maps = [
    cosky.get_emission(freq)[0,:] if freq.to(u.GHz) > 1 * u.GHz else np.zeros(hp.nside2npix(nside))*u.K_RJ for freq in tqdm(nu_centers)
]  # Get free-free maps for each frequency)
co_maps = np.array(co_maps) * u.uK_RJ
co_maps = co_maps - np.mean(co_maps, axis=1, keepdims=True)

In [ ]:
masked_sync_maps = np.array([sync_maps[i] * mask for i in range(len(nu_centers))])
masked_freefree_maps = np.array([freefree_maps[i] * mask for i in range(len(nu_centers))])
masked_cmb_maps = np.array([cmb_maps[i] * mask for i in range(len(nu_centers))])
masked_dust_maps = np.array([dust_maps[i] * mask for i in range(len(nu_centers))])
masked_pt_src_maps = np.array([pt_srcs[i] * mask for i in range(len(nu_centers))])
masked_ame_maps = np.array([ame_maps[i] * mask for i in range(len(nu_centers))])
# masked_tsz_maps = np.array([tsz_maps[i] * mask for i in range(len(nu_centers))])
masked_co_maps = np.array([co_maps[i] * mask for i in range(len(nu_centers))])

In [ ]:
for freq in [30, 44, 70, 100, 143, 217, 353, 545]:
    mock = np.load(f"/projectnb/darkcosmo/dark_photon_project/21cmfast_cache/pyilc_Planck/Planck_{freq}_Cl.npy")
    real = np.load(f"/projectnb/darkcosmo/dark_photon_project/21cmfast_cache/Planck_maps/Planck_{freq}_Cl.npy")
    plt.plot(real/mock[:len(real)], label=f"{freq} GHz")
# plt.plot(mock[:len(real)], label=f"Mock {freq} GHz")
plt.xscale("log")
plt.yscale("log")
plt.ylim(0.1, 5)
plt.xlim(10, 2000)
plt.legend(loc="lower left", fontsize=10)
plt.xlabel(r"$\ell$")
plt.ylabel("Real $C_\ell$/Mock $C_\ell$")

In [ ]:
masked_rms_mean_sync = np.sqrt(np.nanmean(masked_sync_maps**2, axis=1))
masked_rms_mean_freefree = np.sqrt(np.nanmean(masked_freefree_maps**2, axis=1))
masked_rms_mean_cmb = np.sqrt(np.nanmean(masked_cmb_maps**2, axis=1))
masked_rms_mean_dust = np.sqrt(np.nanmean(masked_dust_maps**2, axis=1))
masked_rms_mean_pt_src = np.sqrt(np.nanmean(masked_pt_src_maps**2, axis=1))
masked_rms_mean_ame = np.sqrt(np.nanmean(masked_ame_maps**2, axis=1))
# masked_rms_mean_tsz = np.sqrt(np.nanmean(masked_tsz_maps**2, axis=1))
masked_rms_mean_co = np.sqrt(np.nanmean(masked_co_maps**2, axis=1))

In [ ]:
np.nanmean(np.where(0==mccarthy_mask, np.nan, mccarthy_mask) * real_planck_clean)

In [ ]:
pt_srcs = np.interp(nu_centers, np.geomspace(100, 1000e3, 10) * u.MHz, np.array([5.39787278e-01, 3.58690881e-02, 3.26698519e-03, 4.34741266e-04,
       9.34010089e-05, 3.42318040e-05, 2.00625193e-05, 1.62236209e-05,
       1.58377324e-05, 1.72517065e-05])*1e6
)
planck_nus = np.array([30, 44, 70, 100, 143, 217, 353, 545]) * u.GHz
planck_rms = np.array([1803.777, 685.66275, 345.0852, 369.65762, 278.657, 551.84751, 902.41121, 1371.4603]) * u.uK_RJ
planck_interp_rms = np.interp(nu_centers, planck_nus.to(u.MHz), planck_rms, np.nan, np.nan)


In [ ]:
# plt.loglog(binning.get_effective_ells(), Cls_our_planck * (1000 / prefac)**2, "o", label="Mock Planck maps")
# plt.loglog(binning.get_effective_ells(), Cls_real_planck * (1000 / prefac)**2, "o", label="Real Planck maps")
plt.loglog(binning.get_effective_ells(), mccarthy_Cls, label="McCarthy Maps")
# plt.loglog(radio_bins.get_effective_ells(), radio_Cls * (1000 / prefac)**2, "o", label="Our ILC with mock radio maps")
plt.legend()
plt.xlabel(r"$\ell$")
plt.ylabel(r"$C_{\ell} \, \, [{\rm \mu K_{CMB}^2}]$")
plt.xlim(10, 2048)
plt.ylim(5e-7, 5e-3)

In [ ]:
print("Computing stds so that we know how well the ILC did")
print(rf"Our ILC with mock Planck maps: {np.std(our_planck_clean[mccarthy_mask>0]* 1000 / prefac):.3f} uK")
print(rf"Our ILC with real Planck maps: {np.std(real_planck_clean[mccarthy_mask>0]* 1000 / prefac):.3f} uK")
print(rf"The McCarthy maps: {np.std(mccarthy_map[mccarthy_mask>0]):.3f} uK")


In [ ]:
# plt.plot(nu_centers/1000, dp_rms_uK_RJ, label="Dark Photon")
plt.loglog(nu_centers/1000, (masked_rms_mean_sync
                        + masked_rms_mean_freefree
                        + masked_rms_mean_cmb
                        + masked_rms_mean_dust
                        + masked_rms_mean_pt_src
                        + pt_srcs
                        + masked_rms_mean_ame
                        # + masked_rms_mean_tsz
                        + masked_rms_mean_co
                        )/rms
                        , label="Us")
plt.loglog(nu_centers/1000, planck_interp_rms/rms, label="Planck")

# plt.loglog(nu_centers/1000, (mccarthy_rms_mean_sync
#                         + mccarthy_rms_mean_freefree
#                         + mccarthy_rms_mean_cmb
#                         + mccarthy_rms_mean_cib
#                         + mccarthy_rms_mean_dust
#                         + mccarthy_rms_mean_pt_src
#                         )/rms
#                         , label="McCarthy")
plt.axvspan(30-0.1 * 30, 30 + 0.1 * 30, color='gray', alpha=0.3)
plt.axvspan(44-0.1 * 44, 44 + 0.1 * 44, color='gray', alpha=0.3)
plt.axvspan(70-0.1 * 70, 70 + 0.1 * 70, color='gray', alpha=0.3)
plt.axvspan(100-0.33/2 * 100, 100 + 0.33/2 * 100, color='gray', alpha=0.3)
plt.axvspan(143-0.33/2 * 143, 143 + 0.33/2 * 143, color='gray', alpha=0.3)
plt.axvspan(217-0.33/2 * 217, 217 + 0.33/2 * 217, color='gray', alpha=0.3)
plt.axvspan(353-0.33/2 * 353, 353 + 0.33/2 * 353, color='gray', alpha=0.3)
plt.axvspan(545-0.33/2 * 545, 545 + 0.33/2 * 545, color='gray', alpha=0.3)
plt.axvspan(857-0.33/2 * 857, 857 + 0.33/2 * 857, color='gray', alpha=0.3)
plt.axvspan(266/1000, 1.39, color='gray', alpha=0.3)

plt.legend()
plt.xlabel(r"$\nu$ [GHz]")

In [ ]:
plt.plot(nu_centers/1000, masked_rms_mean_sync, label="sync")
plt.plot(nu_centers/1000, masked_rms_mean_freefree, label="free-free")
plt.plot(nu_centers/1000, masked_rms_mean_cmb, label="CMB")
# plt.plot(nu_centers/1000, masked_rms_mean_cib, label="CIB")
plt.plot(nu_centers/1000, masked_rms_mean_dust, label="Dust")
plt.plot(nu_centers/1000, pt_srcs, label="Point Sources")
plt.plot(nu_centers/1000,  rms, label="Dark Photon")
# plt.plot(nu_centers/1000, planck_interp_rms, label="Planck")
plt.plot(nu_centers/1000, masked_rms_mean_ame, label="AME")
# plt.plot(nu_centers/1000, masked_rms_mean_tsz, label="tSZ")
plt.plot(nu_centers/1000, masked_rms_mean_co, label="CO")
plt.plot(nu_centers/1000, masked_rms_mean_sync
                        + masked_rms_mean_freefree
                        + masked_rms_mean_cmb
                        + masked_rms_mean_dust
                        + masked_rms_mean_pt_src
                        + pt_srcs
                        + masked_rms_mean_ame
                        # + masked_rms_mean_tsz
                        + masked_rms_mean_co
                        , label="Total")
plt.xscale("log")
plt.yscale("log")
plt.axvspan(30, 857 + 0.33/2 * 857, color='gray', alpha=0.3)
plt.axvspan(266/1000, 1.39, color='gray', alpha=0.3)
# plt.xlim(10, 1000)
# plt.ylim(0.005, 500)
plt.legend()
plt.xlabel(r"$\nu$ [GHz]")
plt.ylabel(r"RMS brightness temperature [$\rm \mu K_{RJ}$]")

In [ ]:
hp.nside2resol(1024, arcmin=True)